# yt
> Utilities for Content Creation From YouTube and Local MP4 Videos

This notebook provides utilities for working with video content:

- **YouTube videos**: Fetch transcripts and generate chapters using YouTube URLs
- **Local MP4 files**: Transcribe using OpenAI Whisper and generate chapters using Gemini

Key functions:
- `yt_chapters()`: Generate video summaries and chapter timestamps (works with both YouTube URLs and MP4 files)
- `transcribe()`: Get transcripts from YouTube or transcribe local MP4 files with timestamps

In [ ]:
#|default_exp yt

In [ ]:
#|export
import re
import subprocess
import tempfile
from pathlib import Path
from typing import Optional, Annotated
import typer
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from hamel.gem import gem
import whisper

## YouTube/Video Chapter Creation

Automate chapter creation + description for YouTube videos or local MP4 files

In [ ]:
#|export
def yt_chapters(url_or_path):
    "Generate YouTube Summary and Chapters from a video (YouTube URL or local MP4)."
    
    chapter_prompt="Generate a succinct video summary (1-2 sentences) followed by video chapter timestamps for this video. Format each line of the chapter summaries as 'MM:SS - Chapter Title' (e.g., '02:30 - Introduction'). Start with 00:00. Include all major topics and transitions and be thorough - do not miss any important topics.  For the summary, do not say 'In this video, we will cover the following topics', 'This video discusses..' or anything like that. Instead, reference the main speaker's name if you know it.  If there is a Q&A Section, enumerate individual questions as additional chapters."
    return gem(prompt=chapter_prompt, o=url_or_path, model="gemini-2.5-pro")

This is what it looks like for Antoine's [Late Interaction Talk](https://youtu.be/1x3k0V2IITo):

In [ ]:
chp = yt_chapters('context_rot/context_rot.mp4')
print(chp)

Kelly Hong from Chroma presents the "Context Rot" technical report, demonstrating that Large Language Models' performance is not uniform across input lengths. She reveals through several experiments that performance degrades significantly as input tokens increase, especially when tasks involve semantic ambiguity or distracting information, highlighting the critical need for thoughtful context engineering over simply expanding context windows.

00:00 - Introduction to Kelly Hong and the "Context Rot" Report
00:32 - What is Context Rot?
01:47 - The Rise of Long Context Windows in Frontier Models
02:07 - The Common Assumption: More Context is Always Better
03:33 - Explaining the Needle in a Haystack (NIAH) Benchmark
04:49 - Experiment 1: Adding Ambiguity (Semantic vs. Lexical Matching)
06:10 - Q&A: Clarification on High vs. Low Performance Model Graphs
06:54 - Q&A: Was the Original Needle in a Haystack Benchmark Pointless?
08:08 - Experiment 1: Implications for Real-World Applications
09:

In [ ]:
chp = yt_chapters("https://youtu.be/1x3k0V2IITo")
print(chp)

In this presentation, Antoine Chaffin from LightOn explains the intrinsic limitations of single-vector search models, particularly their struggle with out-of-domain generalization and long contexts due to information loss from pooling. He introduces late-interaction (multi-vector) models as a superior alternative that retains all token-level information and presents his PyLate library, designed to make these powerful models accessible and easy to train.

00:00 - Introduction
00:32 - About the Speaker: Antoine Chaffin
01:40 - How Dense (Single) Vector Search Works
03:07 - Why Single Vector Search is the Go-To for RAG
03:54 - Performance Evaluation with Leaderboards (MTEB)
04:17 - The BEIR Benchmark and Goodhart's Law
05:36 - Limitations Not Captured by Benchmarks: Long Context
06:32 - Limitations Not Captured by Benchmarks: Reasoning-Intensive Retrieval
08:24 - The Intrinsic Flaw of Dense Models: Pooling and Information Compression
10:42 - Why BM25 Remains Competitive
11:32 - Replacing 

In [ ]:
# Works with local MP4 files too
# chp_local = yt_chapters("path/to/your/video.mp4")
# print(chp_local)

## Fetch YouTube Transcript or Transcribe Local MP4

Fetch the YouTube transcript from public videos or transcribe local MP4 files using OpenAI Whisper.

In [ ]:
#|export
def _extract_video_id(url: str) -> Optional[str]:
    """Extract YouTube video ID from various URL formats."""
    for pattern in [r'(?:youtube\.com/watch\?v=|youtu\.be/)([^&\n?#]+)', 
                    r'youtube\.com/embed/([^&\n?#]+)', 
                    r'youtube\.com/v/([^&\n?#]+)']:
        if match := re.search(pattern, url): return match.group(1)
    return url if re.match(r'^[a-zA-Z0-9_-]{11}$', url) else None

def _format_timestamp(seconds: float) -> str:
    """Convert seconds to HH:MM:SS format."""
    h, m, s = int(seconds // 3600), int((seconds % 3600) // 60), int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"
        
def _format_seconds(seconds: float): return f"{int(seconds):d}s"       

def transcribe_local_video(file_path, seconds_only=False):
    "Transcribe local MP4 video using Whisper."
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as temp_audio:
        try:
            # Use ffmpeg to extract audio
            cmd = ["ffmpeg", "-i", str(file_path), "-q:a", "0", "-map", "a", 
                   temp_audio.name, "-y", "-loglevel", "error"]
            subprocess.run(cmd, check=True)
            
            # Load Whisper model and transcribe
            model = whisper.load_model("large-v3-turbo")
            result = model.transcribe(temp_audio.name)
            
            # Format output to match YouTube transcript format
            format_func = _format_seconds if seconds_only else _format_timestamp
            transcript_text = '\n'.join(f"[{format_func(segment['start'])}] {segment['text'].strip()}" 
                                       for segment in result['segments'])
            return transcript_text
        finally:
            # Clean up temp file
            Path(temp_audio.name).unlink(missing_ok=True)

def transcribe(url_or_path, seconds_only=False):
    "Download YouTube transcript or transcribe local video."
    path = Path(url_or_path)
    
    # Check if input is a local file
    if path.exists() and path.suffix.lower() == '.mp4':
        return transcribe_local_video(url_or_path, seconds_only)
    
    # Otherwise treat as YouTube URL
    if not (video_id := _extract_video_id(url_or_path)): 
        raise ValueError(f"Could not extract video ID from '{url_or_path}'")
    try: 
        transcript_data = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])
    except (TranscriptsDisabled, NoTranscriptFound) as e: 
        raise ValueError(f"{str(e)} for video: {video_id}")
    format_func = _format_seconds if seconds_only else _format_timestamp
    transcript_text = '\n'.join(f"[{format_func(e['start'])}] {e['text']}" for e in transcript_data)
    return transcript_text

In [ ]:
t = transcribe("https://youtu.be/1x3k0V2IITo")

In [ ]:
print(t[:500])

[00:00:00] Hello everyone, my name is Chapan and I
[00:00:02] am a research engineer at Leighton and
[00:00:05] today I will detail some of the limits
[00:00:08] of single vector search that have been
[00:00:10] highlighted by recent usages and
[00:00:13] evaluations and then I will introduce
[00:00:16] multi vector models also known as late
[00:00:18] interaction models and how they can
[00:00:21] overcome this and to finish I will
[00:00:24] briefly present the pilot library that
[00:00:26] al


### Local MP4 Transcription

You can also transcribe local MP4 files using OpenAI Whisper:

Note: Requires ffmpeg and openai-whisper installed:
 - macOS: brew install ffmpeg
 - Ubuntu: apt-get install ffmpeg
 - pip install openai-whisper

In [ ]:
t_local = transcribe("_videos/test_video.mp4")
print(t_local[:500])

100%|█████████████████████████████████████| 1.51G/1.51G [00:34<00:00, 47.3MiB/s]
/Users/hamel/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[00:00:00] Hello, this is a super short test recording where I'm going to
[00:00:04] say 1, 2, 3, 4, 5, 6 and say my name, Hamil Hussain.
